# 04 — Retrieval-only evaluation

Compare dense / BM25 / hybrid (RRF) / hybrid+reranker on a 15-question pilot QA set with known ground-truth chunks. Metrics: recall@5, recall@10, MRR.

Outputs the retrieval ablation block of `docs/eval_methodology.md`.

In [1]:
# Path setup: chdir to project root + add to sys.path so imports + .env paths both work.
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside filings-rag/notebooks/.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')


Project root: C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\filings-rag


In [2]:
from pathlib import Path

import json
import os
from dotenv import load_dotenv
load_dotenv(Path(".env"))

from src.retrieval.embeddings import BGEEmbedder
from src.retrieval.vector_store import ChromaStore
from src.retrieval.bm25_index import BM25Index
from src.retrieval.hybrid import reciprocal_rank_fusion
from src.retrieval.reranker import BGEReranker

C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\filings-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load indices and pilot QA set
embedder = BGEEmbedder()
vec_store = ChromaStore(collection="filings", persist_dir=Path(os.environ.get("CHROMA_DIR", "data/vector/chroma")))
bm25 = BM25Index.load(Path(os.environ.get("BM25_PATH", "data/vector/bm25.pkl")))
reranker = BGEReranker()

with Path("data/qa_pilot.jsonl").open(encoding="utf-8") as f:
    pilot = [json.loads(line) for line in f]

print(f"Pilot questions: {len(pilot)}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:  36%|███▌      | 140/393 [00:00<00:00, 1386.72it/s]

Loading weights:  71%|███████   | 279/393 [00:00<00:00, 1166.11it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1187.42it/s]

Pilot questions: 5


In [4]:
# Build chunk_lookup (id -> chunk record) from processed JSONLs
PROCESSED = Path("data/processed")
chunk_lookup = {}
for jsonl in PROCESSED.rglob("*.jsonl"):
    with jsonl.open(encoding="utf-8") as f:
        for line in f:
            c = json.loads(line)
            chunk_lookup[c["chunk_hash"]] = {
                "text": c["text"],
                "metadata": {"ticker": c["ticker"], "year": c["year"], "page": c["page"]},
            }
print(f"Chunks in lookup: {len(chunk_lookup)}")

Chunks in lookup: 19399


In [5]:
def recall_at_k(retrieved_ids, gold_ids, k):
    """Fraction of gold IDs found in top-k retrieved."""
    if not gold_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return sum(1 for g in gold_ids if g in top_k) / len(gold_ids)

def mrr(retrieved_ids, gold_ids):
    """Mean reciprocal rank of the first gold ID in the ranked list."""
    for rank, _id in enumerate(retrieved_ids, start=1):
        if _id in gold_ids:
            return 1.0 / rank
    return 0.0

def aggregate(metrics_per_q):
    return {k: sum(m[k] for m in metrics_per_q) / len(metrics_per_q) for k in metrics_per_q[0]}

In [6]:
# Strategy 1: Dense only
dense_metrics = []
for row in pilot:
    q_vec = embedder.embed([row["question"]])[0]
    hits = vec_store.query(q_vec, k=20)
    ids = [h["id"] for h in hits]
    dense_metrics.append({
        "recall@5": recall_at_k(ids, row["ground_truth_chunk_ids"], 5),
        "recall@10": recall_at_k(ids, row["ground_truth_chunk_ids"], 10),
        "mrr": mrr(ids, row["ground_truth_chunk_ids"]),
    })
print("Dense only:", aggregate(dense_metrics))

Dense only: {'recall@5': 0.6666666666666666, 'recall@10': 0.8, 'mrr': 1.0}


In [7]:
# Strategy 2: BM25 only
bm25_metrics = []
for row in pilot:
    hits = bm25.query(row["question"], k=20)
    ids = [h["id"] for h in hits]
    bm25_metrics.append({
        "recall@5": recall_at_k(ids, row["ground_truth_chunk_ids"], 5),
        "recall@10": recall_at_k(ids, row["ground_truth_chunk_ids"], 10),
        "mrr": mrr(ids, row["ground_truth_chunk_ids"]),
    })
print("BM25 only:", aggregate(bm25_metrics))

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25 only: {'recall@5': 0.5333333333333333, 'recall@10': 0.5333333333333333, 'mrr': 1.0}


In [8]:
# Strategy 3: Hybrid (RRF over top-50 from each)
hybrid_metrics = []
for row in pilot:
    q_vec = embedder.embed([row["question"]])[0]
    dense_hits = vec_store.query(q_vec, k=50)
    bm25_hits = bm25.query(row["question"], k=50)
    fused = reciprocal_rank_fusion([dense_hits, bm25_hits])
    ids = [h["id"] for h in fused][:20]
    hybrid_metrics.append({
        "recall@5": recall_at_k(ids, row["ground_truth_chunk_ids"], 5),
        "recall@10": recall_at_k(ids, row["ground_truth_chunk_ids"], 10),
        "mrr": mrr(ids, row["ground_truth_chunk_ids"]),
    })
print("Hybrid (RRF):", aggregate(hybrid_metrics))

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Hybrid (RRF): {'recall@5': 0.7333333333333333, 'recall@10': 0.8, 'mrr': 1.0}


In [9]:
# Strategy 4: Hybrid + reranker
hyrr_metrics = []
for row in pilot:
    q_vec = embedder.embed([row["question"]])[0]
    dense_hits = vec_store.query(q_vec, k=50)
    bm25_hits = bm25.query(row["question"], k=50)
    fused = reciprocal_rank_fusion([dense_hits, bm25_hits])[:20]
    candidates = [chunk_lookup[h["id"]] for h in fused if h["id"] in chunk_lookup]
    for c, h in zip(candidates, fused):
        c["id"] = h["id"]
    reranked = reranker.rerank(row["question"], candidates, top_k=10)
    ids = [c["id"] for c in reranked]
    hyrr_metrics.append({
        "recall@5": recall_at_k(ids, row["ground_truth_chunk_ids"], 5),
        "recall@10": recall_at_k(ids, row["ground_truth_chunk_ids"], 10),
        "mrr": mrr(ids, row["ground_truth_chunk_ids"]),
    })
print("Hybrid + re-ranker:", aggregate(hyrr_metrics))

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Hybrid + re-ranker: {'recall@5': 0.4666666666666666, 'recall@10': 0.6666666666666667, 'mrr': 0.4666666666666667}


In [10]:
# Summary table for eval_methodology.md
import pandas as pd
table = pd.DataFrame({
    "Dense only": aggregate(dense_metrics),
    "BM25 only": aggregate(bm25_metrics),
    "Hybrid (RRF)": aggregate(hybrid_metrics),
    "Hybrid + re-ranker": aggregate(hyrr_metrics),
}).T
table = table[["recall@5", "recall@10", "mrr"]].round(3)
print(table.to_markdown())
table.to_csv("docs/retrieval_ablation.csv")

|                    |   recall@5 |   recall@10 |   mrr |
|:-------------------|-----------:|------------:|------:|
| Dense only         |      0.667 |       0.8   | 1     |
| BM25 only          |      0.533 |       0.533 | 1     |
| Hybrid (RRF)       |      0.733 |       0.8   | 1     |
| Hybrid + re-ranker |      0.467 |       0.667 | 0.467 |
